# Oversight Arena — TRL GRPO training notebook

Trains a small open-weight overseer LLM to detect malicious peer agents in collaborative coding, against the live OpenEnv HF Space.

**Runtime:** Colab T4 (free tier) for Qwen-2.5-0.5B; A100/L4 for 1.5B+.
**Wall time:** ~30 min for 0.5B / 16 prompts; ~3h for 1.5B / 64 prompts.
**Reproducible:** every seed fixed; held-out eval seeds (9000-9019) disjoint from train seeds (1000-1063).


## 1. Environment setup

In [ ]:
!pip install -q --upgrade trl peft transformers accelerate datasets bitsandbytes
!pip install -q openenv-core>=0.2.1
!pip install -q matplotlib pandas requests

# Clone the repo (read-only — judges can re-run without GitHub auth)
!git clone https://github.com/anikasoni/oversight-arena.git
%cd oversight-arena
!pip install -q -e .

## 2. Sanity-check the live HF Space

In [ ]:
import requests, json

ENV_URL = 'https://anikasoni-oversight-arena.hf.space'

print('health   :', requests.get(f'{ENV_URL}/health').json())
r = requests.post(f'{ENV_URL}/reset', json={'seed': 42, 'difficulty': 0.5}).json()
print('reset    : workers =', r['observation']['workers'])
print('             diff   =', r['observation']['focused_patch_diff'][:200], '...')

s = requests.post(f'{ENV_URL}/step', json={'action': 'flag_worker', 'worker_id': 'W2', 'cwe_tag': 'CWE-89'}).json()
print('step.flag: reward =', s['reward'], 'done =', s['done'])

g = requests.get(f'{ENV_URL}/grader').json()
print('grader   : f1 =', g['f1'], '| tp/fp/fn =', g['tp'], g['fp'], g['fn'])

## 3. Run baseline eval (untrained)

In [ ]:
# Quick baseline: an untrained Qwen-0.5B against 20 held-out seeds.
# Replace with --model Qwen/Qwen2.5-1.5B-Instruct on a real GPU.

import os
os.environ['OVERSIGHT_ENV_URL'] = ENV_URL

!python scripts/train_grpo.py \
    --env-url $OVERSIGHT_ENV_URL \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --n-prompts 4 --num-generations 2 --batch-size 2 \
    --lr 1e-5 --eval-after-train --eval-n 8

## 4. View results

In [ ]:
from IPython.display import Image, display
import json

with open('results/training_summary.json') as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))

for png in ['loss_curve.png', 'reward_curve.png', 'eval_comparison.png']:
    p = f'results/{png}'
    try:
        display(Image(p))
    except FileNotFoundError:
        print('missing:', p)

## 5. (Optional) GPT-4o-mini reference baseline

Compare our trained 1.5B against a frontier API on the same held-out seeds. Skip if no `OPENAI_API_KEY`.

In [ ]:
import os
if os.environ.get('OPENAI_API_KEY'):
    !python scripts/eval_gpt_baseline.py --eval-n 20 --model gpt-4o-mini
    with open('results/eval_gpt_summary.json') as f:
        print(json.load(f))
else:
    print('OPENAI_API_KEY not set — skipping reference baseline.')